In [5]:
from openai import OpenAI
from dotenv import load_dotenv

In [6]:
openai_client = OpenAI()

In [7]:
def llm_prompt(prompt: str) -> str:
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt)
    return response.output_text

In [8]:
llm_prompt("What is the capital of France?")

'The capital of France is **Paris**.'

In [9]:
context ='''
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.
edit on GitHub
#
Cloud alternatives with GPU

Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

    Google Colab
    Kaggle
    Databricks (possibly)

Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
edit on GitHub
#
Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?

When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

'''

In [10]:
prompt=f"""
answer the question based on the following context: {context} \n\n Question: Can i join the course if I just discovered it?
"""

In [11]:
print(prompt)


answer the question based on the following context: 
I just discovered the course. Can I still join?

Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.
edit on GitHub
#
Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?

You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
edit on GitHub
#
What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?

The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on

In [12]:
print(llm_prompt(prompt))

Yes, you can still join the course.

If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [13]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [15]:
documents =[]
base_url = "https://datatalks.club/faq"
for course in courses_raw:
    course_url = f'''{base_url}{course["path"]}'''
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()
    documents.extend(course_data)
len(documents)


1208

In [16]:
documents[800]

{'id': '53afe4bee2',
 'course': 'data-engineering-zoomcamp',
 'section': 'Module 4: Analytics Engineering with dbt',
 'question': 'DBT: Docs Served but Not Accessible via Browser',
 'answer': 'Try removing the `network: host` line in `docker-compose`.'}

In [17]:
question = "Can I join the course if I just discovered it?"

In [18]:
from minsearch import Index
index = Index(
text_fields=['question', 'answer', 'section'],
keyword_fields=['course']
)
index.fit(documents)

In [19]:
search_results = index.search(question, filter_dict={'course': 'llm-zoomcamp'}, boost_dict={'question': 2.0, 'section': 0.5}, num_results=5)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [20]:
def search(question):
    return index.search(question, 
            filter_dict={'course': 'llm-zoomcamp'}, 
            boost_dict={'question': 2.0, 'section': 0.5}, 
            num_results=5
            )

In [21]:
question = "Can I join the course if I just discovered it?"

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [30]:
USER_PROMPT_TEMPLATE = f'''

Question: 
{question}

Context: 
{context}

'''

In [25]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [26]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [32]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question, context=context)
    return prompt.strip()

In [29]:
context

'General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time

In [33]:
prompt = build_prompt(question, search_results)
print(prompt)

Question: 
Can I join the course if I just discovered it?

Context: 
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your proj